# 03 HWW EFT fits for all relevant Wilson coefficients
Uses repo-consistent response extraction and plotting style matched to legacy notebooks (mplhep + CMS-like settings).

## Notebook purpose
Execute H→WW EFT scans using the same response-extraction logic as legacy STXS fits, then evaluate the HWW likelihood.

## Core mathematical chain
1. Extract STXS yields at reweight points for selected WC(s).
2. Fit quadratic response in WC space:
   - 1D: f(c)=a c^2 + b c + d
   - 2D: f(c1,c2)=a c1^2 + b c1 + g c2^2 + h c2 + k c1 c2 + d
3. Convert yield to cross section using legacy normalization:
   σ_fb = 3.594×1000×(yield/sumw_all_noEW).
4. Build μ_pred and evaluate χ²_HWW.
5. Scan WC grid to obtain best-fit and confidence structure.

## Function map
- `stxs_coeffs_fullpath`: coefficient extraction from coffea histograms.
- `scan_hww_1d`: per-operator χ² curve.
- `scan_hww_2d`: pairwise Δχ² surface.
- `run_all_hww_1d_fits`: batch fits over all selected operators.
- `plot_hww_1d`, `plot_hww_2d`: style-consistent visualization.

## Important interpretation caveat
Current prediction is based on STXS proxy bins (221/222/223/224). If your measured HWW region (m_jj>1 TeV) differs, apply a transfer/calibration before claiming final physics limits.

In [ ]:
from pathlib import Path
import json, sys, itertools
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.lines import Line2D
from coffea import util
hep.style.use('CMS')

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p=Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p/marker).exists(): return p
        if p.parent==p: break
        p=p.parent
    raise FileNotFoundError
REPO=find_repo_root(); NBDIR=REPO/'notebooks_hww_fit'
for p in ['/uscms_data/d3/azhou/smeft/analysis','/uscms_data/d3/azhou/smeft/analysis/hbb-coffea']:
    if p not in sys.path: sys.path.append(p)
import stxs_functions as sf
manifest=json.loads((NBDIR/'asset_manifest.json').read_text())
meas=json.loads((NBDIR/'hww_measurement.json').read_text())
coffea_fullpath=manifest['default_coffea_fullpath']
print('coffea:',coffea_fullpath)

In [ ]:
# Operator set scoured from fit_plots.ipynb
SMEFT_OPS = ['cHbox','cHDD','cHj1','cHj3','cHu','cHd','cHudRe','cuWRe','cuBRe','cdWRe','cdBRe','cHW','cHB','cHWB']
SMEFTCPV_OPS = ['cHudIm','cuWIm','cdWIm','cuBIm','cdBIm','cHWtil','cHBtil','cHWBtil']
ALL_OPS = SMEFT_OPS + SMEFTCPV_OPS
print('n ops:',len(ALL_OPS), ALL_OPS)

In [ ]:
def stxs_coeffs_fullpath(coffea_fullpath, operator_list):
    all_h=util.load(coffea_fullpath)['htxs']
    wc_mapping=sf.wc_map_dict(operator_list)
    b1,b2={},{}
    for p,wcl in wc_mapping.items():
        if wcl not in list(all_h.axes['wc']): continue
        y1,y2=sf.get_bin_yields(all_h[{'wc':wcl}]); b1[p]=y1; b2[p]=y2
    pts=[p for p in wc_mapping if p in b1 and p in b2]
    if len(operator_list)==1:
        x=np.array([p[0] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_1D,x,y1,p0=[1,1,1]); c2,_=sf.curve_fit(sf.quad_1D,x,y2,p0=[1,1,1])
    else:
        x1=np.array([p[0] for p in pts]); x2=np.array([p[1] for p in pts]); y1=np.array([b1[p] for p in pts]); y2=np.array([b2[p] for p in pts])
        c1,_=sf.curve_fit(sf.quad_2D,(x1,x2),y1,p0=[1,1,1,1,1,1]); c2,_=sf.curve_fit(sf.quad_2D,(x1,x2),y2,p0=[1,1,1,1,1,1])
    return c1,c2

In [ ]:
def chi2_hww(mu):
    d=mu-meas['mu_obs']; s=meas['sig_up'] if d>=0 else meas['sig_dn']; return (d/s)**2

def scan_hww_1d(op, sigma_sm_hww_fb, wc_min=-20,wc_max=20,n=401,mg_sigma_pb=3.594):
    sumw=util.load(coffea_fullpath)['sumw_all_noEW'].value
    c1,c2=stxs_coeffs_fullpath(coffea_fullpath,[op])
    rows=[]
    for x in np.linspace(wc_min,wc_max,n):
        sig=(mg_sigma_pb*1000*sf.quad_1D(x,*c1)/sumw)+(mg_sigma_pb*1000*sf.quad_1D(x,*c2)/sumw)
        mu=sig/sigma_sm_hww_fb
        rows.append({'op':op,'wc':float(x),'sig_proxy_fb':float(sig),'mu_pred':float(mu),'chi2_hww':float(chi2_hww(mu))})
    return pd.DataFrame(rows)

def scan_hww_2d(op1,op2,sigma_sm_hww_fb,wc_min=-10,wc_max=10,n=121,mg_sigma_pb=3.594):
    sumw=util.load(coffea_fullpath)['sumw_all_noEW'].value
    c1,c2=stxs_coeffs_fullpath(coffea_fullpath,[op1,op2])
    rows=[]
    g=np.linspace(wc_min,wc_max,n)
    for x in g:
      for y in g:
        sig=(mg_sigma_pb*1000*sf.quad_2D((x,y),*c1)/sumw)+(mg_sigma_pb*1000*sf.quad_2D((x,y),*c2)/sumw)
        mu=sig/sigma_sm_hww_fb
        rows.append({'op1':op1,'op2':op2,'wc1':float(x),'wc2':float(y),'chi2_hww':float(chi2_hww(mu))})
    return pd.DataFrame(rows)

In [ ]:
def run_all_hww_1d_fits(sigma_sm_hww_fb, ops=ALL_OPS, wc_min=-20,wc_max=20,n=401):
    out={}
    for op in ops:
        try: out[op]=scan_hww_1d(op,sigma_sm_hww_fb,wc_min,wc_max,n)
        except Exception as e: print('skip',op,e)
    return out

def summarize_1d(best_dict):
    rows=[]
    for op,df in best_dict.items():
        i=df['chi2_hww'].idxmin(); r=df.loc[i]
        rows.append({'op':op,'best_wc':r['wc'],'chi2_min':r['chi2_hww']})
    return pd.DataFrame(rows).sort_values('chi2_min')

In [ ]:
def plot_hww_1d(df, title=None):
    plt.figure(figsize=(6,4))
    plt.plot(df['wc'],df['chi2_hww'],label=df['op'].iloc[0])
    cmin=df['chi2_hww'].min(); plt.axhline(cmin+1,color='r',ls='--',label='1$\sigma$')
    plt.xlabel('Wilson coefficient'); plt.ylabel(r'$\chi^2$')
    plt.ylim(0,max(10,cmin+4)); plt.legend(); plt.title(title or f"HWW 1D: {df['op'].iloc[0]}")
    plt.tight_layout()

In [ ]:
def plot_hww_2d(df):
    op1,op2=df['op1'].iloc[0],df['op2'].iloc[0]
    c1=sorted(df['wc1'].unique()); c2=sorted(df['wc2'].unique())
    X,Y=np.meshgrid(c1,c2); Z=np.zeros((len(c2),len(c1)))
    for i,y in enumerate(c2):
      for j,x in enumerate(c1): Z[i,j]=df[(df.wc1==x)&(df.wc2==y)]['chi2_hww'].iloc[0]
    Z=Z-Z.min()
    plt.figure(figsize=(7,5))
    cmap=plt.cm.viridis.copy(); cmap.set_over('white')
    m=plt.pcolormesh(X,Y,Z,cmap=cmap,vmin=0,vmax=10); cb=plt.colorbar(m,label=r'$\Delta\chi^2$'); cb.set_ticks([0,2.30,5.99,10])
    plt.contour(X,Y,Z,levels=[2.30],colors='red'); plt.contour(X,Y,Z,levels=[5.99],colors='black',linestyles='dotted')
    plt.xlabel(op1); plt.ylabel(op2); plt.title(f'HWW 2D: {op1} vs {op2}')
    plt.tight_layout()

In [ ]:
# Run all 1D fits (set sigma_sm_hww_fb before running):
# ALL_1D = run_all_hww_1d_fits(sigma_sm_hww_fb=1.0)
# SUM_1D = summarize_1d(ALL_1D); SUM_1D.head()
# plot_hww_1d(ALL_1D['cHW'])

# Run all pairwise 2D fits (can be expensive):
# pairs=list(itertools.combinations(ALL_OPS,2))
# SCAN2D={(a,b):scan_hww_2d(a,b,sigma_sm_hww_fb=1.0,n=81) for a,b in pairs}
# plot_hww_2d(SCAN2D[('cHW','cHWtil')])